# Project purpose on application ML engineering

| Domain / Module | Modeling Approach | Objective / Target | Key Notes & Constraints |
| :--- | :--- | :--- | :--- |
| **Fraud prediction** | Anomaly / integrity detection | Detect data integrity violations | Impossible to model true fraud (no labels, no financial fields). Checks rules like refused > requested, staff_present > total_staff, and departure < arrival. |
| **Service performance** | ✅ Regression / ranking | Predict a Service Health Score | — |
| **Staff analytics** | ✅ Regression + association | Predict morale / attendance | — |
| **Operational risk** | ✅ Regression / classification | Predict your `risk_score` or `risk_level` | — |
| **Patient management** | ✅ Classification | Predict `patient_churn` | — |

## Import library

In [15]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib
import json
import joblib

from pathlib import Path

pd.set_option('display.max_columns', None)

## Load & Feature Engineering

In [16]:
BASE_PATH = Path.cwd().parent
DATA_PATH = BASE_PATH / "database" / "hospital_database_with_pred_churn.parquet"

# get the access data
df = pd.read_parquet(DATA_PATH)

# normalize typo
if "staff_absens" in df.columns and "staff_absent" not in df.columns:
    df["staff_absent"] = df["staff_absens"]

# Feature engineering
df["arrival_date"] = pd.to_datetime(df["arrival_date"], errors="coerce")
df["departure_date"] = pd.to_datetime(df["departure_date"], errors="coerce")

# separate arrival data into the features
df["arrival_year"] = df["arrival_date"].dt.year
df["arrival_month"] = df["arrival_date"].dt.month
df["arrival_week"] = df["arrival_date"].dt.isocalendar().week.astype(int)
df["arrival_dow"] = df["arrival_date"].dt.dayofweek
df["is_weekend"] = df["arrival_dow"].isin([5,6]).astype(int)

# Capacity pressure features
df["capacity_pressure"] = df["patients_request"] / (df["available_beds"] + 1)
df["capacity_pressure_level"] = pd.cut(df["capacity_pressure"], [-np.inf, 1, 2, 4, np.inf], labels=["Low", "Moderate", "High", "Critical"])

## Feature engineering second phrase

In [ ]:
# Admission / refusal / attendance rates features
df["admission_rate_row"] = df["patients_admitted"] / df["patients_request"].replace(0, np.nan)
df["refusal_rate_row"] = df["patients_refused"] / df["patients_request"].replace(0, np.nan)
df["staff_attendance_rate"] = df["staff_present"] / df["total_staff"].replace(0, np.nan)

# Operational risk score
los_min, los_max = df["length_of_stay"].min(), df["length_of_stay"].max()
df["_dissat"] = (100 - df["satisfaction"]) / 100
df["_los_risk"] = (df["length_of_stay"] - los_min) / max(los_max - los_min, 1)
df["_refusal_pressure"] = df["refusal_rate_row"].fillna(0).clip(0, 1)
df["_morale_risk"] = (100 - df["staff_morale"]) / 100
df["risk_score"] = (0.40*df["_dissat"] + 0.25*df["_los_risk"] + 0.20*df["_refusal_pressure"] + 0.15*df["_morale_risk"]) * 100
df["risk_level"] = pd.cut(df["risk_score"], [-1, 30, 60, 100], labels=["Low", "Medium", "High"])

# Service health score (regression target for modeling)
svc = df.groupby("service").agg(
    avg_sat=("satisfaction", "mean"),
    churn=("patient_churn", "mean"),
    adm_rate=("admission_rate_row", "mean"),
    att=("staff_attendance_rate", "mean")
)
svc["health_score"] = (0.35*svc.avg_sat/100 + 0.25*(1-svc.churn)
                      + 0.20*svc.adm_rate + 0.20*svc.att) * 100
df = df.merge(svc["health_score"], on="service", how="left")
    
print(df.shape)
df.head()

(10000, 40)


,patient_id,name,age,arrival_date,departure_date,service,satisfaction,month,week,available_beds,patients_request,patients_admitted,patients_refused,patient_satisfaction,staff_morale,total_staff,staff_present,staff_absens,length_of_stay,patient_churn,best_pred_churn,age_bin,staff_absent,arrival_year,arrival_month,arrival_week,arrival_dow,is_weekend,capacity_pressure,capacity_pressure_level,admission_rate_row,refusal_rate_row,staff_attendance_rate,_dissat,_los_risk,_refusal_pressure,_morale_risk,risk_score,risk_level,health_score
0,PAT-09484753,Richard Rodriguez,24,2025-03-16,2025-03-22,surgery,61,3,3,27,66.0,27.0,39.0,63.0,72.0,25,0.0,25.0,6,1,1,19-35,25.0,2025,3,11,6,1,2.357143,High,0.409091,0.590909,0.000000,0.39,0.384615,0.590909,0.28,41.233566,Medium,74.355371
1,PAT-f0644084,Shannon Walker,6,2025-12-13,2025-12-14,surgery,83,12,2,40,26.0,26.0,0.0,96.0,56.0,25,23.0,2.0,1,0,0,0-18,2.0,2025,12,50,5,1,0.634146,Low,1.000000,0.000000,0.920000,0.17,0.000000,0.000000,0.44,13.400000,Low,74.355371
2,PAT-ac6162e4,Julia Torres,24,2025-06-29,2025-07-05,general_medicine,83,6,5,40,103.0,40.0,63.0,73.0,52.0,28,23.0,5.0,6,1,1,19-35,5.0,2025,6,26,6,1,2.512195,High,0.388350,0.611650,0.821429,0.17,0.384615,0.611650,0.48,35.848394,Medium,52.384121
3,PAT-3dda2bb5,Crystal Johnson,32,2025-10-12,2025-10-23,emergency,81,10,2,28,169.0,28.0,141.0,75.0,64.0,39,37.0,2.0,11,1,1,19-35,2.0,2025,10,41,6,1,5.827586,Critical,0.165680,0.834320,0.948718,0.19,0.769231,0.834320,0.36,48.917160,Medium,48.986184
4,PAT-08591375,Garrett Lin,25,2025-02-18,2025-02-25,ICU,76,2,3,20,21.0,20.0,1.0,82.0,89.0,34,0.0,34.0,7,0,0,19-35,34.0,2025,2,8,1,0,1.000000,Low,0.952381,0.047619,0.000000,0.24,0.461538,0.047619,0.11,23.740842,Low,106.574244


## Shared Preprocessing Helpers

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, mean_absolute_error, r2_score, mean_squared_error

DROP_COLS = ["patient_id","name","arrival_date","departure_date",
               "month","week","age_bin"] 

def build_preprocessor(X: pd.DataFrame):
    """Build a preprocessor for the ML model."""
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    num_pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                            ("sc", StandardScaler())])
    cat_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("enc", OneHotEncoder(handle_unknown="ignore"))])

    return ColumnTransformer([("num", num_pipe, num_cols),
                              ("cat", cat_pipe, cat_cols)]), num_cols, cat_cols

def evaluate_classifier(name, y_true, y_pred, y_prob=None):
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_prob) if y_prob is not None else np.nan
    }

def evaluate_regressor(name, y_true, y_pred):
    return {
        "model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error((y_true - y_pred))),
        "R2": r2_score(y_true, y_pred)
    }